# Virtual Cell Challenge 2026 — CPU-first baseline, evaluation and final predictions

**Inputs:** public 2025 H1 training data (`adata_Training.h5ad`), 2026 validation controls (`A/B/C`), and the separately downloadable final controls (`D/E/F`).

**Model:** raw-count perturbation signature transfer with regularized (shrunken) gene-wise log fold changes. Train on public perturbed vs non-targeting cells, validate on a strictly held-out portion of public data, then predict 2026 contexts from their non-targeting controls. No GPU required. This is a **baseline, not STATE, GEARS, or PertAdapt**.

**Critical limitations:** (1) the provided 2026 contexts contain controls only; organizer-held-out 2026 responses cannot be scored locally. Use the official `vcc` validation submission for actual 2026 score. (2) 2025 H1-only held-out-cell evaluation is **not** zero-shot cross-cell-type evaluation. (3) targets missing from 2025 training default to no perturbation effect; their predictions will likely be weak. (4) 360,000 × 18,533 output is large, so plan for a **64 GiB RAM VM and ≥50–100 GB free local disk**; monitor actual usage.

**Execution order:** setup → inspect GCS/download → read schemas → train/cache → local public-data evaluation → validation prediction/package → final prediction/package. Change only paths and annotation overrides in the configuration cell. The final submission is **never sent automatically**.

Official references: [VCC submission requirements](https://vcc-cli-wiki.virtualcellchallenge.org/) · [2026 metrics](https://github.com/ArcInstitute/cell-eval2/blob/main/docs/vcc2026_metrics/vcc2026-metrics.md) · [2025 Atlas](https://github.com/ArcInstitute/arc-virtual-cell-atlas/blob/main/virtual-cell-challenge/README.md).


## 0. Dependencies and compute checks

In [ ]:
# Run once in the notebook's actual Python kernel, not an unrelated `pip` binary.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'anndata>=0.10,<0.13', 'pandas', 'numpy', 'scipy',
                       'scikit-learn', 'h5py', 'psutil', 'matplotlib'])
# Restart the kernel only if the environment says you must.

In [ ]:
from pathlib import Path
import os, json, math, gc, time, shutil, subprocess, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
import anndata as ad
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
print('Python:', __import__('sys').version.split()[0], 'RAM (GiB):', round(psutil.virtual_memory().total/2**30,1))
print('Google Cloud storage CLI:', shutil.which('gcloud'))

## 1. Configuration: Google Cloud + validation/final controls

In [ ]:
# On Google Compute Engine, use a VM service account with read permission for GCS.
# Other machines: authenticate once using `gcloud auth login` outside the notebook.
PROJECT = 'virtual-cell-challenge-2026'
TRAIN_GCS = 'gs://2026-vcc-llsa/train/adata_Training.h5ad'
# 2025 training CSV is optional: use it to inspect intended target counts; labels come from AnnData.obs.
TRAIN_COUNTS_GCS = 'gs://2026-vcc-llsa/train/pert_counts_Training.csv'

ROOT = Path.home() / 'vcc2026_cpu'
TRAIN_DIR = ROOT/'train'; VAL_DIR=ROOT/'vcc_2026_controls'; FINAL_DIR=ROOT/'vcc_2026_controls_final'
ARTIFACTS=ROOT/'artifacts'; PRED_DIR=ROOT/'predictions'
for d in (TRAIN_DIR,VAL_DIR,FINAL_DIR,ARTIFACTS,PRED_DIR): d.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=TRAIN_DIR/'adata_Training.h5ad'

# If files have already been uploaded into the notebook's local machine, specify that directory:
# VAL_DIR = Path('/some/path/vcc_2026_controls')
# FINAL_DIR = Path('/some/path/vcc_2026_controls_final')

# Annotation overrides — leave None for automatic detection; check the schema output before running training.
PERT_COL=None  # e.g. 'gene', 'target_gene', 'perturbation'
CONTROL_LABEL=None # e.g. 'non-targeting', 'control', 'NTC'
REPLICATE_COL=None # optional batch/replicate metadata for later evaluation

SEED=2026
N_VALID_PER_GROUP=40
MIN_CELLS_PER_GROUP=15
SHRINKAGE=0.65       # global strength of source perturbation signatures, 0..1
PSEUDOCOUNT=0.5      # regularization for lowly expressed genes in mean-count LFC
CLIP_LOG_FOLD=1.0    # cap absolute ln fold change for transport robustness
CELLS_PER_PERT=400
MAX_CELL_COUNTS=1_000_000
WRITE_PREDICTIONS=False   # flip to True after inspection and successful smoke test
N_PERT_SMOKE=3            # set None for complete 300-target outputs
print(ROOT)

## 2. Stage training data and inspect 2026 bundles

In [ ]:
def cloud_copy(uri, dest, optional=False):
    if dest.exists():
        print('Already cached:', dest, round(dest.stat().st_size/1e9,2), 'GB'); return True
    if not shutil.which('gcloud'):
        if optional: print('Optional file not staged; gcloud not installed:',uri); return False
        raise RuntimeError('Install gcloud CLI and authenticate on your VM or set local TRAIN_PATH')
    cmd=['gcloud','storage','cp',uri,str(dest)]
    print('Running:', ' '.join(cmd)); r=subprocess.run(cmd)
    if r.returncode and not optional: raise RuntimeError(f'GCS download failed: {uri}')
    return r.returncode==0

# Uncomment to download 15.5GB only after checking available disk:
# cloud_copy(TRAIN_GCS, TRAIN_PATH)
# cloud_copy(TRAIN_COUNTS_GCS, TRAIN_DIR/'pert_counts_Training.csv', optional=True)

def inspect_bundle(directory):
    mpath=directory/'manifest.json'
    if not mpath.exists():
        print('Not staged:', directory); return None
    m=json.loads(mpath.read_text())
    print('Bundle:',directory,'partition:',m.get('partition'),'contexts:',m.get('contexts'),
          'genes:',m.get('n_genes'),'targets:',m.get('n_constructs'))
    for c in m['contexts']:
        p=directory/f'context_{c}.h5ad'
        if not p.exists(): print('Missing:',p); continue
        a=ad.read_h5ad(p, backed='r')
        print(c, a.shape,'obs:',list(a.obs.columns),'sparse:',type(a.X).__name__)
        a.file.close()
    return m

VAL_MANIFEST=inspect_bundle(VAL_DIR)
FINAL_MANIFEST=inspect_bundle(FINAL_DIR)
print('If bundles missing: stage the uploaded validation files or run `vcc datasets download controls` / `controls-final`, then unzip into the specified directories.')

### Download the challenge bundles using the official CLI (terminal)

Install the official CLI according to its [guide](https://vcc-cli-wiki.virtualcellchallenge.org/), authenticate as a registered participant, and run:

```bash
vcc datasets download controls -d ~/vcc2026_cpu/vcc_2026_controls
vcc datasets download controls-final -d ~/vcc2026_cpu/vcc_2026_controls_final
```

Unzip the archive(s), keeping `context_*.h5ad`, `manifest.json`, `pert_counts.csv`, and `gene_names.csv` together directly in their respective folders. Keep validation and final panels separate; final labels should be D/E/F and its target list is different.


## 3. Inspect the public 2025 training schema before touching counts

In [ ]:
assert TRAIN_PATH.exists(), f'Put public 2025 data here, or run cloud_copy above: {TRAIN_PATH}'
train=ad.read_h5ad(TRAIN_PATH, backed='r')
print('Public train shape:',train.shape)
print('obs columns:',train.obs.columns.tolist())
print('example obs:\n',train.obs.head(5).to_string())
print('first genes:',train.var_names[:12].tolist())
print('X type:',type(train.X), 'dtype:',train.X.dtype)
print('layers:',list(train.layers.keys()))
print('obs example value frequencies:')
for col in train.obs.columns:
    if any(k in col.lower() for k in ('gene','pert','target','condition','control')):
        print(col, train.obs[col].astype(str).value_counts().head(8).to_dict())

## 4. Select correct labels and RAW expression values

In [ ]:
def infer_label_column(obs, override=None):
    if override is not None:
        if override not in obs: raise KeyError(f'Perturbation column not found: {override}')
        return override
    for name in ('target_gene','perturbation','gene','condition','target','pert'):
        if name in obs.columns: return name
    raise ValueError('Cannot identify perturbation annotation. Set PERT_COL manually after inspecting obs.')

def infer_control(values, override=None):
    values=pd.Series(values).astype(str)
    if override is not None:
        assert (values==override).any(), f'Control label absent: {override}'
        return override
    candidates=['non-targeting','non_targeting','NTC','ctrl','control','Control','non-targeting control','NT','non-targeting_ctrl']
    found=[k for k in candidates if (values==k).any()]
    if len(found)!=1: raise ValueError(f'Control label ambiguous. Detected {found}; set CONTROL_LABEL based on obs counts')
    return found[0]

PERT_COL_SELECTED=infer_label_column(train.obs, PERT_COL)
labels=train.obs[PERT_COL_SELECTED].astype(str).to_numpy()
CONTROL_SELECTED=infer_control(labels,CONTROL_LABEL)
print('Selected:',PERT_COL_SELECTED, 'Control:',repr(CONTROL_SELECTED))
print('Control cells:',int(np.sum(labels==CONTROL_SELECTED)))
print('Perturbations:', len(np.unique(labels[labels!=CONTROL_SELECTED])))

# Diagnose expression units; this notebook assumes raw integer single-cell counts.
probe=train.X[:min(64,train.n_obs)]
if sp.issparse(probe): probe=probe.data
else: probe=np.asarray(probe).ravel()
assert np.isfinite(probe).all() and (probe>=0).all(), 'Invalid expression matrix'
assert np.all(np.abs(probe-np.rint(probe))<1e-5), ('X appears normalized/log transformed; identify raw count layer before proceeding')
assert len(set(train.var_names))==train.n_vars, 'Duplicate gene names require explicit mapping before training'
print('Small sample looks like raw counts. Verify full dataset uses raw counts.')

## 5. Memory-efficient training: mean control and mean perturbation signatures

In [ ]:
def mean_rows_backed(adata, indices, block=256):
    """Sparse-safe mean expression; indexes may refer to a backed AnnData."""
    indices=np.sort(np.asarray(indices,dtype=np.int64))
    out=np.zeros(adata.n_vars,dtype=np.float64)
    if not len(indices): raise ValueError('Empty subset')
    for i in range(0,len(indices),block):
        # sorted index read avoids some h5ad backed indexing problems
        chunk=adata.X[indices[i:i+block], :]
        if sp.issparse(chunk): out += np.asarray(chunk.sum(axis=0)).ravel()
        else: out += np.asarray(chunk,dtype=np.float64).sum(axis=0)
    return out / len(indices)

def compute_bank(adata, lbls, control_label, train_groups=None, split_for_eval=False):
    """Hold out cells within each known perturbation for an optimistic public-data sanity check."""
    rng=np.random.default_rng(SEED)
    groups=np.unique(lbls) if train_groups is None else np.asarray(train_groups)
    idx_map={}
    for group in groups:
        ids=np.flatnonzero(lbls==group)
        if split_for_eval and len(ids)>max(MIN_CELLS_PER_GROUP+N_VALID_PER_GROUP,N_VALID_PER_GROUP+1):
            take=rng.choice(ids,size=N_VALID_PER_GROUP,replace=False)
            keep=np.setdiff1d(ids,take,assume_unique=True)
            idx_map[group]=(keep,take)
        else: idx_map[group]=(ids,np.array([],dtype=np.int64))
    baseline=mean_rows_backed(adata,idx_map[control_label][0])
    effects={}; n_kept={}
    for group,(kept,_) in idx_map.items():
        if group==control_label or len(kept)<MIN_CELLS_PER_GROUP: continue
        mean=mean_rows_backed(adata,kept)
        # raw count means -> regularized log fold change; no repeated dense cell matrices
        effect=np.log((mean+PSEUDOCOUNT)/(baseline+PSEUDOCOUNT))
        effects[str(group)]=np.clip(effect,-CLIP_LOG_FOLD,CLIP_LOG_FOLD).astype(np.float32)
        n_kept[str(group)]=len(kept)
    return baseline,effects,idx_map,n_kept

# Public-data holdout: validates within-source prediction only (NOT cross-cell-type transfer).
start=time.time()
base_eval,effects_eval,split_indices,kept_n=compute_bank(train,labels,CONTROL_SELECTED,split_for_eval=True)
print('Trained signatures:',len(effects_eval),'time (min):',round((time.time()-start)/60,1))
print('Examples:',list(effects_eval)[:10])

## 6. Evaluate on held-out public 2025 cells (proxy validation only)

In [ ]:
# Compare predicted mean expression of held-out cells with control-only and signature-transfer predictions.
# This DOES NOT provide an official 2026 challenge score.
results=[]
for pert,effect in effects_eval.items():
    val_indices=split_indices[pert][1]
    if len(val_indices)==0: continue
    y_true=mean_rows_backed(train,val_indices)
    pred_zero=base_eval
    pred_effect=base_eval*np.exp(SHRINKAGE*effect)
    results.append({'target_gene':pert,'n_valid':len(val_indices),
                    'mse_control_log1p':float(np.mean((np.log1p(pred_zero)-np.log1p(y_true))**2)),
                    'mse_transfer_log1p':float(np.mean((np.log1p(pred_effect)-np.log1p(y_true))**2)),
                    'corr_effect':float(np.corrcoef((y_true-base_eval), (pred_effect-base_eval))[0,1]) if np.std(y_true-base_eval)>0 and np.std(pred_effect-base_eval)>0 else np.nan})
proxy_eval=pd.DataFrame(results)
print(proxy_eval[['mse_control_log1p','mse_transfer_log1p','corr_effect']].describe())
print('Transfer beats control (fraction):', (proxy_eval.mse_transfer_log1p<proxy_eval.mse_control_log1p).mean())
proxy_eval.to_csv(ARTIFACTS/'public_2025_heldout_proxy_metrics.csv', index=False)
assert len(proxy_eval)>0, 'No validation groups; review 2025 annotations or reduce N_VALID_PER_GROUP'


## 7. Refit on all 2025 cells; save a reusable small model

In [ ]:
start=time.time()
base_all,effects_all,_,n_train=compute_bank(train,labels,CONTROL_SELECTED,split_for_eval=False)
np.savez_compressed(ARTIFACTS/'signature_bank_2025.npz',
                    genes=np.asarray(train.var_names.astype(str)),
                    perts=np.asarray(list(effects_all)),
                    effects=np.stack(list(effects_all.values())),
                    control_mean=base_all, shrinkage=np.array([SHRINKAGE]))
print('Saved', ARTIFACTS/'signature_bank_2025.npz','size MB',round((ARTIFACTS/'signature_bank_2025.npz').stat().st_size/1e6,2))
train.file.close(); del train; gc.collect()
print('Elapsed minutes',round((time.time()-start)/60,1))

## 8. Align public source features to 2026 genes and inspect coverage

In [ ]:
bank=np.load(ARTIFACTS/'signature_bank_2025.npz', allow_pickle=True)
source_genes=bank['genes'].astype(str)
source_perts=bank['perts'].astype(str)
source_effects=bank['effects']
source_gene_index={x:i for i,x in enumerate(source_genes)}
source_pert_index={x:i for i,x in enumerate(source_perts)}

def bundle_metadata(directory):
    manifest=json.loads((directory/'manifest.json').read_text())
    gene_list=pd.read_csv(directory/'gene_names.csv').iloc[:,0].astype(str).tolist()
    pert_list=pd.read_csv(directory/'pert_counts.csv').iloc[:,0].astype(str).tolist()
    assert len(gene_list)==len(set(gene_list)), 'Duplicate target gene axis'
    assert len(pert_list)==len(set(pert_list)), 'Duplicate perturbation panel'
    assert len(gene_list)==manifest['n_genes'], 'Manifest gene count mismatch'
    assert len(pert_list)==manifest['n_constructs'], 'Manifest perturbation count mismatch'
    return manifest,gene_list,pert_list

def report_coverage(directory):
    manifest,genes,perts=bundle_metadata(directory)
    print(directory, 'contexts:',manifest['contexts'], 'gene intersection:',sum(x in source_gene_index for x in genes), '/',len(genes),
          'perturbations with trained signature:',sum(p in source_pert_index for p in perts),'/',len(perts))
    missing=[x for x in perts if x not in source_pert_index]
    print('Unseen target examples:',missing[:30])
    return missing
if (VAL_DIR/'manifest.json').exists(): report_coverage(VAL_DIR)
if (FINAL_DIR/'manifest.json').exists(): report_coverage(FINAL_DIR)

## 9. Generate cell-level predictions (sparse raw integer counts)

For each perturbed condition, sample 400 real control-cell profiles **with replacement**, multiply observed nonzero counts by the learned fold-change vector, then use stochastic rounding. This retains some natural cell-to-cell heterogeneity while remaining sparse. It cannot generate positive counts in genes that are zero in the sampled control cell (a modeling limitation). No controls appear in the submitted predictions. Unknown perturbations currently receive unchanged control samples. This is a deliberately conservative baseline.


In [ ]:
def make_predicted_rows(X_ctrl, sample_idx, log_effect, strength, rng):
    """Sparse CSR transformation; no dense cells x genes materialization."""
    subset=X_ctrl[sample_idx,:].tocsr(copy=True).astype(np.float32)
    if log_effect is not None:
        factor=np.exp(np.clip(strength*log_effect,-CLIP_LOG_FOLD,CLIP_LOG_FOLD))
        subset.data *= factor[subset.indices]
    # Stochastic rounding preserves the count expectation better than np.rint at low counts.
    lo=np.floor(subset.data)
    subset.data=(lo+(rng.random(len(lo))<(subset.data-lo))).astype(np.float32)
    subset.eliminate_zeros()
    totals=np.asarray(subset.sum(axis=1)).ravel()
    if np.any(totals>MAX_CELL_COUNTS):
        raise ValueError('Cell count cap violated; inspect input and scaling before continuing')
    assert np.isfinite(subset.data).all() and np.all(subset.data>=0)
    return subset

def generate_context(directory, context, outpath, only_n_perts=None):
    manifest,genes,perts=bundle_metadata(directory)
    if only_n_perts is not None: perts=perts[:only_n_perts]
    a=ad.read_h5ad(directory/f'context_{context}.h5ad',backed='r')
    assert a.n_vars==len(genes), 'Wrong context gene axis length'
    a_genes=a.var_names.astype(str).tolist()
    if a_genes!=genes:
        raise ValueError('Context gene axis differs from gene_names.csv; align explicitly before generating')
    assert a.obs['context'].astype(str).eq(context).all(), 'Context labels do not match source data'
    assert a.obs['target_gene'].astype(str).eq(manifest['control_label']).all(), 'Not a pure control bundle'
    src_idx=np.asarray([source_gene_index.get(g,-1) for g in genes],dtype=np.int64)
    rng=np.random.default_rng(SEED+sum(map(ord,context)))
    # Each slice is read from disk once, then retained in RAM for the current context (~200MB compressed disk).
    X=a.X[:,:]
    if not sp.issparse(X): X=sp.csr_matrix(X)
    else: X=X.tocsr()
    out_parts=[]; out_labels=[]
    missing=[]
    for j,p in enumerate(perts):
        sel=rng.choice(a.n_obs,size=CELLS_PER_PERT,replace=True)
        if p not in source_pert_index:
            delta=None; missing.append(p)
        else:
            delta=np.zeros(len(genes),dtype=np.float32)
            match=src_idx>=0
            delta[match]=source_effects[source_pert_index[p],src_idx[match]]
        part=make_predicted_rows(X,sel,delta,SHRINKAGE,rng)
        out_parts.append(part)
        out_labels.extend([p]*CELLS_PER_PERT)
        if (j+1)%50==0: print(context,'generated',j+1,'/',len(perts),'targets')
    X_out=sp.vstack(out_parts,format='csr'); X_out.eliminate_zeros()
    obs=pd.DataFrame({'target_gene':pd.Categorical(out_labels),
                      'context':pd.Categorical([context]*len(out_labels))},
                     index=[f'{context}_{i:07d}' for i in range(len(out_labels))])
    result=ad.AnnData(X=X_out,obs=obs,var=pd.DataFrame(index=pd.Index(genes,name='gene_name')))
    print(context, 'shape:',result.shape, 'nnz:',result.X.nnz,'unseen perturbations:',len(missing))
    if outpath is not None:
        result.write_h5ad(outpath,compression='gzip')
        print('Wrote',outpath,round(outpath.stat().st_size/1e9,2),'GB')
    a.file.close();del a,X,X_out,out_parts;gc.collect()
    return result if outpath is None else None


## 10. Smoke-test a small validation prediction first

In [ ]:
assert (VAL_DIR/'manifest.json').exists(), 'Stage validation controls first'
m,genes,perts=bundle_metadata(VAL_DIR)
first_context=m['contexts'][0]
mini=generate_context(VAL_DIR,first_context,outpath=None,only_n_perts=N_PERT_SMOKE)
assert mini.n_obs==CELLS_PER_PERT*N_PERT_SMOKE
assert mini.n_vars==len(genes)
print(mini.obs['target_gene'].value_counts().head())
print('Cells integer:',np.all(mini.X.data==np.rint(mini.X.data)))
print('Max cell total:',np.asarray(mini.X.sum(axis=1)).max())
del mini;gc.collect()

## 11. Generate complete validation predictions (A/B/C)

In [ ]:
def generate_panel(directory, outfile, allow_overwrite=False):
    manifest,genes,perts=bundle_metadata(directory)
    assert len(manifest['contexts'])==3
    if outfile.exists() and not allow_overwrite: raise FileExistsError(f'Protecting existing output {outfile}')
    context_files=[]
    for ctx in manifest['contexts']:
        p=PRED_DIR/f'{manifest["partition"]}_{ctx}_prediction.h5ad'
        generate_context(directory,ctx,outpath=p,only_n_perts=None)
        context_files.append(p)
    # Requires RAM for all context prediction matrices; use a 64–128 GiB VM if this step OOMs.
    ads=[ad.read_h5ad(p) for p in context_files]
    combo=ad.concat(ads,axis=0,join='exact',merge='same',index_unique=None)
    assert combo.n_obs==len(manifest['contexts'])*len(perts)*CELLS_PER_PERT
    combo.X=combo.X.tocsr();combo.X.eliminate_zeros()
    combo.write_h5ad(outfile,compression='gzip')
    print('Complete panel:',outfile,combo.shape,'nnz:',combo.X.nnz)
    del ads,combo;gc.collect()
    return outfile

VAL_PRED=PRED_DIR/'validation_predictions_ABC.h5ad'
if WRITE_PREDICTIONS:
    generate_panel(VAL_DIR,VAL_PRED)
else:
    print('DRY-RUN MODE: set WRITE_PREDICTIONS=True after inspecting coverage and smoke test')

## 12. Validate packaging and obtain **real 2026 validation score**

In [ ]:
def prep_command(prediction_file, directory, vcc_out):
    cmd=['vcc','prep',str(prediction_file),'-g',str(directory/'gene_names.csv'),
         '--perts',str(directory/'pert_counts.csv'),'-o',str(vcc_out)]
    print('\nRun in your VM terminal:\n', ' '.join(cmd))
    return cmd
if VAL_PRED.exists():
    prep_command(VAL_PRED,VAL_DIR,PRED_DIR/'validation_predictions_ABC.vcc')
print('Use vcc prep ... --dry-run first. Then vcc prep to package.')
print('If validation submissions are open: vcc submit ... --wait (requires your explicit action).')
print('Do NOT call this a local 2026 ground-truth evaluation: the organizer holds out the real perturbation cells.')

## 13. Final dataset (D/E/F): repeat using new manifest and targets

In [ ]:
FINAL_PRED=PRED_DIR/'final_predictions_DEF.h5ad'
if (FINAL_DIR/'manifest.json').exists():
    fm,fg,fp=bundle_metadata(FINAL_DIR)
    assert fm['contexts']==['D','E','F'], f'Unexpected final contexts: {fm["contexts"]}'
    print('Final panel:',len(fp),'targets /',len(fg),'genes; contexts:',fm['contexts'])
    report_coverage(FINAL_DIR)
    if WRITE_PREDICTIONS:
        generate_panel(FINAL_DIR,FINAL_PRED)
else:
    print('Stage the independent vcc_2026_controls_final bundle first; do not copy validation metadata.')
if FINAL_PRED.exists():
    prep_command(FINAL_PRED,FINAL_DIR,PRED_DIR/'final_predictions_DEF.vcc')
print('Recommended: vcc prep <final.h5ad> -g <final gene_names.csv> --perts <final pert_counts.csv> -o final.vcc')
print('Inspect official CLI version (>=0.3.0) and use vcc whoami. Submit only after a complete prep succeeds.')

## 14. Scientific next steps and major pitfalls

1. **Cross-context generalization:** 2025 H1-only within-dataset holdout overestimates true 2026 performance. Add other public cell types, construct the same signature bank per source context, and use leave-one-cell-type-out validation. Avoid training on 2026 held-out perturbation data.
2. **Unseen knockdowns:** Calculate overlap with each panel (cell above). A no-effect fallback is a weak predictor for new perturbation genes; add gene-embedding or pathway-based regression only after establishing the baseline.
3. **Cell-level scoring:** This baseline modifies existing nonzero control counts and can miss DE genes absent in sampled controls. Consider a sparse count generator with novel positive entries and cross-context calibration if the official six-metric score indicates this bottleneck.
4. **Data identity:** The 2025 public H1 dataset is not your previous K562 file. Verify gene symbols and perturbation column, raw count units, source license, control labels, and possible duplicate identifiers before training.
5. **Cost and reproducibility:** Stop VMs when idle. Check free RAM and disk before generating 360,000 predicted cells. Store your compressed signature bank separately from the large training file. Seed randomness and do not overwrite prior submissions by default.
6. **Official format:** Exactly 400 predicted cells per perturbation in each of the 3 contexts; raw nonnegative integer counts, 18,533 genes, no controls, sparse representation, maximum cell sum 1,000,000; package all contexts together using `vcc prep`.

**Notebooks don't verify the official biological score without hidden experimental measurements.** A live validation submission reports the official score; final test is blind. Source: [VCC CLI guide](https://vcc-cli-wiki.virtualcellchallenge.org/).
